# F0 — Setup, Validação da Arquitetura Dory-4B e Baselines

**Objetivo da Fase F0 (conforme `VISAO_GERAL.md` e `ESPEC_ANTIGRAVITY.md`):**
1. Detectar a GPU disponível (`PROFILE = "a100_80" | "a100_40"`), fixar versões de dependências e montar o Google Drive (`/content/drive/MyDrive/coder-comprimido/`) com `HF_HOME` persistente.
2. Executar a suíte de testes unitários (`pytest tests/test_critical.py`) antes de qualquer avaliação.
3. Gerar o split fixo `data/lcb_quick.json` com os 200 problemas mais recentes do LiveCodeBench v6 (`release_v6`).
4. Carregar `nvidia/NVIDIA-Nemotron-Labs-Dory1.0-4B-BF16` via `lib/dory.py`, validar a troca dinâmica de `n_recurrent_loops` $\in \{1, 2, 4\}$ (confirmando mudança de saída e escalonamento de latência) e inspecionar o **Logit Lens por loop**.
5. Executar o **Smoke Test (`SMOKE = True`)** (~5 exemplos) e estimar o tempo e custo em Compute Units (CU) antes do run completo.
6. Rodar `eval_quick` e `eval_gate` nos **4 baselines**:
   - `Dory-4B` com `1`, `2` e `4` loops recorrentes;
   - `Nanbeige/Nanbeige4.2-3B` (referência externa com mesmo orçamento de tokens e opcionalmente orçamento do model card).
7. Registrar todas as métricas em `results/results.jsonl` e salvar o gráfico inicial da **Fronteira de Pareto** em `results/plots/`.

In [ ]:
# 1. Verificação de Hardware e Perfil de Execução (Regra 1.1)
!nvidia-smi --query-gpu=name,memory.total --format=csv

import os
import sys
from pathlib import Path

# Garante que a raiz do repositório está no sys.path
REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
    os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from lib.hw import PROFILE, get_hw_config, print_hardware_summary

hw_info = print_hardware_summary()
eval_hw_cfg = get_hw_config("eval")
print("Configuração de avaliação para o perfil ativo:", eval_hw_cfg)

In [ ]:
# 2. Instalação com Versões Fixadas (Regra 2) e Persistência no Google Drive (Seção 1.2)
!pip install -q \
    "transformers==4.49.0" \
    "accelerate==1.4.0" \
    "trl==0.15.2" \
    "peft==0.14.0" \
    "bitsandbytes==0.45.3" \
    "datasets==3.3.2" \
    "evalplus==0.3.1" \
    "matplotlib==3.10.0" \
    "pytest==8.3.4" \
    "safetensors==0.5.3"

from lib.io import DEFAULT_DRIVE_ROOT, setup_drive_and_cache

PATHS = setup_drive_and_cache(
    drive_root=DEFAULT_DRIVE_ROOT,
    local_fallback_root=REPO_ROOT,
)
for k, v in PATHS.items():
    print(f"{k:12s}: {v}")

RESULTS_JSONL = PATHS["results"] / "results.jsonl"
PLOTS_DIR = PATHS["plots"]
EVAL_CACHE_DIR = PATHS["eval_cache"]
LCB_QUICK_PATH = PATHS["data"] / "lcb_quick.json"

In [ ]:
# 3. Execução Obrigatória dos Testes Unitários dos Módulos Críticos (Seção 6)
import subprocess

ret = subprocess.run([sys.executable, "-m", "pytest", "-v", "tests/test_critical.py"], check=False)
if ret.returncode != 0:
    ret = subprocess.run([sys.executable, "-m", "unittest", "-v", "tests/test_critical.py"], check=True)
assert ret.returncode == 0, "Os testes unitários de lib/ falharam! Corrija antes de prosseguir."

In [ ]:
# 4. Construção / Carregamento do Split Fixo `data/lcb_quick.json` (200 problemas mais recentes do LCB v6)
from lib.evals import build_lcb_quick_split

lcb_quick_items = build_lcb_quick_split(output_path=LCB_QUICK_PATH, n_problems=200)
# Mantém cópia também no diretório local data/lcb_quick.json do repositório
local_lcb_path = REPO_ROOT / "data" / "lcb_quick.json"
if local_lcb_path.resolve() != LCB_QUICK_PATH.resolve():
    local_lcb_path.parent.mkdir(parents=True, exist_ok=True)
    local_lcb_path.write_text(LCB_QUICK_PATH.read_text(encoding="utf-8"), encoding="utf-8")

print(f"Split LCB Quick pronto: {len(lcb_quick_items)} problemas.")
if lcb_quick_items:
    print("Mais recente:", lcb_quick_items[0].get("question_id"), lcb_quick_items[0].get("contest_date"))
    print("200º mais recente:", lcb_quick_items[-1].get("question_id"), lcb_quick_items[-1].get("contest_date"))

In [ ]:
# 5. Carregamento do Dory-4B e Validação de `n_recurrent_loops` em {1, 2, 4} (Seção 2.2)
import torch
from lib.dory import DORY_HF_REPO, load_dory, logit_lens_per_loop, set_dory_loops
from lib.gen import format_chat_prompt, generate

dory_model, dory_tokenizer = load_dory(
    DORY_HF_REPO,
    n_loops=2,
    kv_cache_mode="per_loop",
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

validation_prompt = format_chat_prompt(
    dory_tokenizer,
    "Write a Python function `fibonacci(n)` that returns the n-th Fibonacci number (0-indexed).",
    enable_thinking=True,
)

loop_validation_outputs = {}
for k_loops in (1, 2, 4):
    set_dory_loops(dory_model, k_loops, kv_cache_mode="per_loop")
    out_k = generate(
        dory_model,
        [validation_prompt],
        max_new_tokens=256,
        temperature=0.0,
        n_loops=k_loops,
        tokenizer=dory_tokenizer,
        batch_size=1,
    )[0]
    loop_validation_outputs[k_loops] = out_k
    print(
        f"[n_loops={k_loops}] latency={out_k.latency_s:.3f}s | "
        f"cot_tokens={out_k.n_cot_tokens} | total_tokens={out_k.n_total_tokens}"
    )
    print("  Trecho inicial:", repr(out_k.text[:140]))

In [ ]:
# 6. Inspeção de Logit Lens por Loop (1, 2, 3 e 4) nas últimas 5 posições do prompt (Seção 2.2)
lens_report = logit_lens_per_loop(
    dory_model,
    dory_tokenizer,
    validation_prompt,
    n_loops=4,
    top_k=5,
    last_n_positions=5,
)

for loop_info in lens_report:
    print(f"\n=== Loop Recorrente #{loop_info['loop_idx']} ===")
    for pos in loop_info["positions"]:
        top_pairs = [
            f"{repr(tok)} ({prob:.3f})"
            for tok, prob in zip(pos["top_tokens"], pos["top_probs"])
        ]
        print(
            f"  pos={pos['position_idx']:4d} | in={repr(pos['input_token']):12s} -> "
            + ", ".join(top_pairs)
        )

In [ ]:
# 7. Smoke Test (`SMOKE = True`, 5 exemplos) + Estimativa de Custo em Compute Units (Regra 6)
import time
from lib.evals import eval_quick
from lib.metrics import estimate_compute_cost

SMOKE = True
SMOKE_N = 5

t_smoke_start = time.perf_counter()
smoke_summary = eval_quick(
    dory_model,
    dory_tokenizer,
    run_id="F0_dory_L2_smoke",
    phase="F0",
    model_name=DORY_HF_REPO,
    checkpoint="base",
    n_loops=2,
    lcb_quick_path=LCB_QUICK_PATH,
    max_new_tokens=eval_hw_cfg["max_new_tokens"],
    batch_size=eval_hw_cfg["batch_size"],
    cache_dir=EVAL_CACHE_DIR,
    results_path=RESULTS_JSONL,
    smoke=True,
    smoke_n=SMOKE_N,
)
smoke_elapsed = time.perf_counter() - t_smoke_start
print("Resultado Smoke (Dory L=2):", smoke_summary)

# Total de exemplos para os 3 loops do Dory em eval_quick (164 HE+ + 200 LCB_quick = 364 por loop)
total_quick_items_all_baselines = (164 + 200) * 4
cost_est = estimate_compute_cost(
    smoke_elapsed_s=smoke_elapsed,
    smoke_n_items=SMOKE_N * 2,
    full_n_items=total_quick_items_all_baselines,
    cu_per_hour=12.0,
    confirm_threshold_cu=30.0,
)

In [ ]:
# 8. Execução Completa e Retomável dos Baselines Dory-4B (1, 2 e 4 loops) em `eval_quick` e `eval_gate`
# Altere RUN_FULL_EVAL = True após verificar a estimativa de Compute Units acima.
from lib.evals import eval_gate

RUN_FULL_EVAL = True
CONFIRM_OVER_30_CU = False  # Altere para True caso cost_est['requires_confirmation'] seja True

if RUN_FULL_EVAL:
    if cost_est["requires_confirmation"] and not CONFIRM_OVER_30_CU:
        raise RuntimeError(
            f"Custo estimado ({cost_est['estimated_cu']:.1f} CU) > 30 CU. "
            "Defina CONFIRM_OVER_30_CU = True para confirmar a execução completa."
        )

    for k_loops in (1, 2, 4):
        run_id = f"F0_dory_L{k_loops}"
        set_dory_loops(dory_model, k_loops, kv_cache_mode="per_loop")

        print(f"\n>>> Executando eval_quick para {run_id} (n_loops={k_loops})...")
        eval_quick(
            dory_model,
            dory_tokenizer,
            run_id=run_id,
            phase="F0",
            model_name=DORY_HF_REPO,
            checkpoint="base",
            n_loops=k_loops,
            lcb_quick_path=LCB_QUICK_PATH,
            max_new_tokens=eval_hw_cfg["max_new_tokens"],
            batch_size=eval_hw_cfg["batch_size"],
            cache_dir=EVAL_CACHE_DIR,
            results_path=RESULTS_JSONL,
            smoke=False,
        )

        print(f"\n>>> Executando eval_gate para {run_id} (n_loops={k_loops})...")
        eval_gate(
            dory_model,
            dory_tokenizer,
            run_id=run_id,
            phase="F0",
            model_name=DORY_HF_REPO,
            checkpoint="base",
            n_loops=k_loops,
            max_new_tokens=eval_hw_cfg["max_new_tokens"],
            batch_size=eval_hw_cfg["batch_size"],
            cache_dir=EVAL_CACHE_DIR,
            results_path=RESULTS_JSONL,
            smoke=False,
        )

In [ ]:
# 9. Baseline de Referência Externa: `Nanbeige/Nanbeige4.2-3B` (Seção 2.1)
# Avaliado com o MESMO orçamento de tokens do Dory (4096) para comparação justa, e opcionalmente
# com orçamento estendido (8192) recomendado pelo model card.
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer
from lib.dory import NANBEIGE_HF_REPO

RUN_NANBEIGE_EXTENDED_BUDGET = False

if RUN_FULL_EVAL:
    # Libera VRAM do Dory antes de carregar o Nanbeige4.2-3B
    del dory_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    nanbeige_tokenizer = AutoTokenizer.from_pretrained(NANBEIGE_HF_REPO, trust_remote_code=True)
    if nanbeige_tokenizer.pad_token is None:
        nanbeige_tokenizer.pad_token = nanbeige_tokenizer.eos_token

    nanbeige_model = AutoModelForCausalLM.from_pretrained(
        NANBEIGE_HF_REPO,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True,
    )
    nanbeige_model.eval()

    # 9a. Mesmo max_new_tokens do Dory (comparação justa na mesma régua)
    eval_quick(
        nanbeige_model,
        nanbeige_tokenizer,
        run_id="F0_nanbeige_fair",
        phase="F0",
        model_name=NANBEIGE_HF_REPO,
        checkpoint="base",
        n_loops=None,
        lcb_quick_path=LCB_QUICK_PATH,
        max_new_tokens=eval_hw_cfg["max_new_tokens"],
        batch_size=eval_hw_cfg["batch_size"],
        cache_dir=EVAL_CACHE_DIR,
        results_path=RESULTS_JSONL,
        smoke=False,
    )
    eval_gate(
        nanbeige_model,
        nanbeige_tokenizer,
        run_id="F0_nanbeige_fair",
        phase="F0",
        model_name=NANBEIGE_HF_REPO,
        checkpoint="base",
        n_loops=None,
        max_new_tokens=eval_hw_cfg["max_new_tokens"],
        batch_size=eval_hw_cfg["batch_size"],
        cache_dir=EVAL_CACHE_DIR,
        results_path=RESULTS_JSONL,
        smoke=False,
    )

    # 9b. Opcional: orçamento estendido de geração (card recommendation)
    if RUN_NANBEIGE_EXTENDED_BUDGET:
        eval_quick(
            nanbeige_model,
            nanbeige_tokenizer,
            run_id="F0_nanbeige_card",
            phase="F0",
            model_name=NANBEIGE_HF_REPO,
            checkpoint="base",
            n_loops=None,
            lcb_quick_path=LCB_QUICK_PATH,
            max_new_tokens=8192,
            batch_size=max(1, eval_hw_cfg["batch_size"] // 2),
            cache_dir=EVAL_CACHE_DIR,
            results_path=RESULTS_JSONL,
            smoke=False,
        )

In [ ]:
# 10. Tabela Resumo e Gráfico Inicial de Pareto em `results/plots/` (Seção 3.4)
import shutil
from lib.metrics import compute_pareto_frontier, plot_pareto
from lib.registry import load_results

f0_records = load_results(RESULTS_JSONL, phase="F0")
print(f"Total de registros F0 em {RESULTS_JSONL}: {len(f0_records)}\n")

header = f"{'run_id':18s} | {'eval':6s} | {'bench':14s} | {'loops':5s} | {'pass@1':8s} | {'cot_tok':8s} | {'lat(s)':7s} | {'pass1/1k_tok':12s}"
print(header)
print("-" * len(header))
for r in f0_records:
    loops_str = str(r["n_loops"]) if r.get("n_loops") is not None else "-"
    print(
        f"{r['run_id']:18s} | {r['eval']:6s} | {r['bench']:14s} | {loops_str:5s} | "
        f"{r['pass1']:8.4f} | {r['tokens_cot_mean']:8.1f} | {r['latency_mean_s']:7.2f} | "
        f"{r['acerto_por_1k_tokens']:12.4f}"
    )

for bench_name in ("lcb_v6_quick", "humaneval+", "lcb_v6_full", "mbpp+", "scicode"):
    bench_rows = [r for r in f0_records if r.get("bench") == bench_name]
    if not bench_rows:
        continue
    out_plot = PLOTS_DIR / f"F0_pareto_{bench_name.replace('+', 'plus')}.png"
    plot_pareto(
        bench_rows,
        out_plot,
        bench_filter=bench_name,
        baseline_phase="F0",
        title=f"F0 Baselines — Fronteira de Pareto ({bench_name})",
    )
    print(f"Gráfico salvo em: {out_plot}")

# Sincroniza results.jsonl e plots para a árvore local do repositório caso estejam no Drive
local_results_dir = REPO_ROOT / "results"
if local_results_dir.resolve() != PATHS["results"].resolve() and RESULTS_JSONL.exists():
    local_results_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(RESULTS_JSONL, local_results_dir / "results.jsonl")
    local_plots_dir = local_results_dir / "plots"
    local_plots_dir.mkdir(parents=True, exist_ok=True)
    for p in PLOTS_DIR.glob("*.png"):
        shutil.copy2(p, local_plots_dir / p.name)